# Nemotron-3-Nano-30B LoRA Fine-Tuning (No Unsloth)

Uses standard HuggingFace `transformers` + `peft` + `trl` stack.

**Why NOT Unsloth:** Unsloth is not available offline and does NOT support the hybrid NemotronH architecture (Mamba-2 + Transformer + MoE).

## Architecture: NemotronH (Hybrid)
- 52 layers: 23 Attention + 23 Mamba-2 + 6 MoE
- 30B total params, 3.5B active
- LoRA must target BOTH Attention AND Mamba-2 modules

In [ ]:
import subprocess, sys, os
from pathlib import Path

def resolve_python_path(target_dir):
    for pth_file in Path(target_dir).glob("*.pth"):
        with pth_file.open() as fp:
            relpath = fp.read()
            rel_pack_path = (pth_file.parent / relpath)
            if rel_pack_path.exists():
                print(f"append {rel_pack_path}")
                sys.path.append(str(rel_pack_path))

# Install from offline packages
offline_dir = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages"
Nvidia_whl_dir = "/kaggle/input/datasets/manish756/nvidia-wheel/packages"
target_dir = "/kaggle/working/packages"

os.makedirs(target_dir, exist_ok=True)
resolve_python_path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script")

if os.path.exists(offline_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index",
        "--find-links", offline_dir,
        "--target", target_dir,
        "datasets", "trl"
    ])
if os.path.exists(Nvidia_whl_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index",
        "--find-links", Nvidia_whl_dir,
        "--target", target_dir,
        "datasets", "trl"
    ])
    print("Installed from offline packages")

sys.path.append(target_dir)
resolve_python_path(target_dir)

import datasets
print(f"datasets version: {datasets.__version__}")

In [ ]:
# Install flash attention if available
flash_attn_whl = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
if os.path.exists(flash_attn_whl):
    !pip install --no-index {flash_attn_whl}
else:
    print("Flash attention wheel not found, skipping")

## 1. Load Training Data

In [ ]:
import json
import os
import torch
from datasets import Dataset

# Load metric-aligned CoT data
for candidate in [
    '/kaggle/input/datasets/manish756/nemotron-dataset/train_cot_v3_metric_aligned.jsonl',
    '/kaggle/input/datasets/manish756/nemotron-dataset/train_cot2.0.jsonl',
    '/kaggle/input/datasets/manish756/nemotron-dataset/train_cot.jsonl',
    'train_cot_v3_metric_aligned.jsonl',
    'train_cot2.0.jsonl',
    'train_cot.jsonl',
]:
    if os.path.exists(candidate):
        JSONL_FILE = candidate
        break

data = []
with open(JSONL_FILE, 'r') as f:
    for line in f:
        data.append(json.loads(line))

print(f"Loaded {len(data)} examples from {JSONL_FILE}")
print(f"Roles: {[m['role'] for m in data[0]['messages']]}")

has_system = any(m['role'] == 'system' for m in data[0]['messages'])
has_think = '<think>' in data[0]['messages'][-1]['content']
print(f"System prompt: {has_system}, <think> tags: {has_think}")

## 2. Load Model (Standard HuggingFace — No Unsloth)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Model path (local Kaggle model or HF hub)
MODEL_NAME = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
if not os.path.exists(MODEL_NAME):
    MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

MAX_SEQ_LENGTH = 8192
LORA_RANK = 32

# 4-bit quantization config for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading model from: {MODEL_NAME}")
print(f"This may take a few minutes...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)

# Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()

print(f"Model loaded!")
print(f"Model type: {model.config.model_type}")
print(f"Vocab size: {model.config.vocab_size}")

## 3. Add LoRA Adapters (peft)

Target BOTH Transformer attention AND Mamba-2 layers:
- Attention (23 layers): `q_proj`, `k_proj`, `v_proj`, `o_proj`
- Mamba-2 (23 layers): `in_proj`, `out_proj`
- MLP/MoE: `up_proj`, `down_proj`

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

# LoRA config targeting ALL layer types
lora_config = LoraConfig(
    r=LORA_RANK,                      # Rank 32 (max allowed)
    lora_alpha=64,                     # 2x rank
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        # Transformer Attention (23 layers)
        "q_proj", "k_proj", "v_proj", "o_proj",
        # Mamba-2 SSM (23 layers) — CRITICAL
        "in_proj", "out_proj",
        # MLP / MoE experts
        "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

print(f"\nLoRA config:")
print(f"  Rank: {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Target modules: {lora_config.target_modules}")

## 4. Prepare Dataset

In [ ]:
import re
import numpy as np

dataset = Dataset.from_list(data)

def format_example(example):
    messages = example['messages']
    
    # Remove system prompt if present (evaluation doesn't use one)
    formatted_messages = [m for m in messages if m['role'] != 'system']
    
    # Add <think> tags if not present
    assistant_msg = formatted_messages[-1]
    if '<think>' not in assistant_msg['content']:
        content = assistant_msg['content']
        boxed_match = re.search(r'(\\boxed\{.*?\})\s*$', content)
        if boxed_match:
            reasoning = content[:boxed_match.start()].strip()
            boxed_answer = boxed_match.group(1)
            formatted_messages[-1] = {
                'role': 'assistant',
                'content': f"<think>\n{reasoning}\n</think>\n{boxed_answer}"
            }
    
    # Apply tokenizer's chat template
    try:
        text = tokenizer.apply_chat_template(
            formatted_messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    except Exception:
        parts = []
        for m in formatted_messages:
            parts.append(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>")
        text = '\n'.join(parts)
    
    return {'text': text}

dataset = dataset.map(format_example, num_proc=4)

# Token length check
print("=== Token Length Validation ===")
sample_tokens = tokenizer(dataset[0]['text'], return_tensors='pt')
print(f"Sample: {sample_tokens['input_ids'].shape[1]} tokens")

# Quick check on a sample
sample_lengths = []
for i in range(min(100, len(dataset))):
    toks = tokenizer(dataset[i]['text'], return_tensors='pt')
    sample_lengths.append(toks['input_ids'].shape[1])
sample_lengths = np.array(sample_lengths)
print(f"Sample (n=100): min={sample_lengths.min()}, max={sample_lengths.max()}, mean={sample_lengths.mean():.0f}")
print(f"All within {MAX_SEQ_LENGTH}: {(sample_lengths <= MAX_SEQ_LENGTH).all()}")

print(f"\nFormatted text (first 500 chars):")
print(dataset[0]['text'][:500])

## 5. Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,      # Effective batch = 16
    warmup_steps=50,
    num_train_epochs=3,
    learning_rate=2e-5,
    bf16=True,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=4,
    packing=True,
    args=training_args,
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"Training loss: {trainer_stats.training_loss:.4f}")

## 6. Save LoRA Adapter & Create Submission

In [ ]:
import zipfile

ADAPTER_DIR = "./lora_adapter"

# Save only the LoRA adapter (not the full model)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to {ADAPTER_DIR}")
print(f"Files:")
for f in os.listdir(ADAPTER_DIR):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    print(f"  {f}: {size/1024/1024:.2f} MB")

# Verify adapter_config.json exists
assert os.path.exists(os.path.join(ADAPTER_DIR, 'adapter_config.json')), \
    "adapter_config.json MISSING — submission will fail!"
print("\nadapter_config.json: EXISTS")

# Package as submission.zip
SUBMISSION_PATH = "submission.zip"
with zipfile.ZipFile(SUBMISSION_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(ADAPTER_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, ADAPTER_DIR)
            zf.write(file_path, arcname)

zip_size = os.path.getsize(SUBMISSION_PATH)
print(f"\nsubmission.zip: {zip_size/1024/1024:.2f} MB")

# Show contents
with zipfile.ZipFile(SUBMISSION_PATH, 'r') as zf:
    for info in zf.infolist():
        print(f"  {info.filename}: {info.file_size/1024:.1f} KB")

## Done!

Upload `submission.zip` to the competition. It contains:
- `adapter_config.json` (required)
- `adapter_model.safetensors` (LoRA weights)
- Tokenizer files

### Key differences from Unsloth version:
- Uses `AutoModelForCausalLM` + `BitsAndBytesConfig` (4-bit QLoRA)
- Uses `peft.get_peft_model()` instead of `FastLanguageModel.get_peft_model()`
- Uses `prepare_model_for_kbit_training()` for proper gradient setup
- Gradient checkpointing via `TrainingArguments` (not Unsloth's custom impl)